# Selected depth-limited CIFAR-10 model

This notebook contains `UltraWideScaledTail10`, its layer audit, and the training entry point.
The companion [training script](train_ultrawidescaledtail10_repro.py) defaults to eight seeds and 750 epochs per seed.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import asdict

import pandas as pd
import torch
from IPython.display import display

from train_ultrawidescaledtail10_repro import (
    MODEL_KEY,
    MODEL_NAME,
    TrainConfig,
    UltraWideScaledTail10,
    build_config,
    count_param_layers,
    n_params,
    run_seed_sweep,
    weighted_layer_rows,
)

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Selected model: {MODEL_NAME} ({MODEL_KEY})")


## Selected model

`UltraWideScaledTail10` had the highest mean selected validation accuracy among the three retained architecture comparisons.
It uses a wide convolutional trunk followed by a residual tail at reduced spatial resolution.

The project depth budget counts convolutional and linear modules.
It excludes normalization, pooling, activation, gating, residual addition, stochastic depth, and scalar residual parameters.


In [ ]:
model = UltraWideScaledTail10()
model.eval()

with torch.no_grad():
    dummy_out = model(torch.zeros(2, 3, 32, 32))

model_audit = pd.DataFrame(
    [
        {
            "model": MODEL_NAME,
            "weighted_layers": count_param_layers(model),
            "params_m": n_params(model) / 1_000_000,
            "dummy_output_shape": tuple(dummy_out.shape),
            "within_budget": count_param_layers(model) <= 10,
        }
    ]
)

display(model_audit.style.format({"params_m": "{:.2f}"}))
assert count_param_layers(model) == 10
assert tuple(dummy_out.shape) == (2, 10)


In [ ]:
layer_table = pd.DataFrame(weighted_layer_rows(model))
display(layer_table)


## Earlier selection evidence

The embedded table contains selected validation checkpoints from the earlier 500-epoch runs.
The final eight-seed, 750-epoch results appear in `summary.csv` at the repository root.
Architecture and checkpoint selection can bias validation estimates upward. The final CSV contains no official test accuracy.


In [ ]:
SELECTED_MODEL_SEED_RESULTS = [
    {
        "seed": 42,
        "selected_oos_val": 96.94,
        "best_val_raw": 96.88,
        "best_val_ema": 96.94,
        "selected_epoch": 459,
        "params_m": 21.337656,
        "layers": 10,
        "source_run": "bw500_ultrawidescaledtail10_s42",
    },
    {
        "seed": 43,
        "selected_oos_val": 97.06,
        "best_val_raw": 97.06,
        "best_val_ema": 97.02,
        "selected_epoch": 481,
        "params_m": 21.337656,
        "layers": 10,
        "source_run": "bw500_ultrawidescaledtail10_s43",
    },
    {
        "seed": 44,
        "selected_oos_val": 96.90,
        "best_val_raw": 96.90,
        "best_val_ema": 96.90,
        "selected_epoch": 486,
        "params_m": 21.337656,
        "layers": 10,
        "source_run": "bw500_ultrawidescaledtail10_s44",
    },
]

seed_df = pd.DataFrame(SELECTED_MODEL_SEED_RESULTS)
summary_df = pd.DataFrame(
    [
        {
            "model": MODEL_NAME,
            "seeds": len(seed_df),
            "mean_oos_val": seed_df["selected_oos_val"].mean(),
            "std_oos_val": seed_df["selected_oos_val"].std(ddof=1),
            "max_oos_val": seed_df["selected_oos_val"].max(),
            "params_m": seed_df["params_m"].iloc[0],
            "layers": seed_df["layers"].iloc[0],
        }
    ]
)

display(
    seed_df.style.format(
        {
            "selected_oos_val": "{:.2f}",
            "best_val_raw": "{:.2f}",
            "best_val_ema": "{:.2f}",
            "params_m": "{:.2f}",
        }
    )
)
display(
    summary_df.style.format(
        {
            "mean_oos_val": "{:.2f}",
            "std_oos_val": "{:.2f}",
            "max_oos_val": "{:.2f}",
            "params_m": "{:.2f}",
        }
    )
)


## Eight-seed, 750-epoch run

The training script uses seeds 42 through 49 and writes per-seed checkpoints and logs under `repro_runs_ultrawidescaledtail10_8seed/`.
It also writes `summary.csv` and `summary.json` there.

Run `python train_ultrawidescaledtail10_repro.py` after reviewing its top-of-file configuration.
For an execution check, reduce both the seed count and training work. Use the optional smoke settings below.


In [ ]:
REPRO_CONFIG = build_config()

config_df = pd.DataFrame([asdict(REPRO_CONFIG)]).T.rename(columns={0: "value"})
display(config_df)

print("Terminal command:")
print("python train_ultrawidescaledtail10_repro.py")


In [ ]:
RUN_FULL_REPRODUCIBILITY_TRAINING = False

if RUN_FULL_REPRODUCIBILITY_TRAINING:
    sweep_results, sweep_summary = run_seed_sweep(REPRO_CONFIG)
    display(pd.DataFrame(sweep_results)[["seed", "selected_checkpoint", "selected_val_acc", "selected_epoch", "best_val_raw", "best_val_ema", "wall_sec"]])
    display(pd.DataFrame([sweep_summary]))
else:
    print("Full training is disabled in the notebook. Use the terminal command above, or set RUN_FULL_REPRODUCIBILITY_TRAINING=True.")


## Optional smoke check

This check exercises training, data loading, and checkpoint writing. Its limited training work cannot reproduce the reported accuracy.


In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    smoke_config = TrainConfig(
        seeds=(42,),
        epochs=2,
        batch_size=128,
        num_workers=0,
        max_train_batches=4,
        use_amp=False,
        output_dir="./smoke_ultrawidescaledtail10",
    )
    smoke_results, smoke_summary = run_seed_sweep(smoke_config)
    display(pd.DataFrame(smoke_results)[["seed", "selected_val_acc", "selected_epoch", "wall_sec"]])
else:
    print("Smoke test is disabled.")


## Recorded result

The model has ten counted weighted layers and 21,337,656 parameters.
The final CSV records 96.84% mean selected validation accuracy across eight seeds, with sample standard deviation 0.10 percentage points.
